#  C++ Primer : Struct
# 1. Grouping data
* A `struct` is a type of your own making, holding several values together

```
struct vec {
  double x;
  double y;
};
```

* It defines a _type_ called `vec`; it does not create any variable. Note the `;` after the closing brace
* Using it
```
vec p;                    // a vec variable, members unspecified
vec q = { 1.0, 2.0 };     // x = 1.0, y = 2.0
p.x = 3.0;                // members are reached with .
printf("%f\n", q.y);
```
* A `struct` variable _is_ the data, not a reference to it. `p = q;` copies both members, and the two variables stay independent afterwards. This is unlike Python, where assigning an object copies a reference
* Likewise, passing a `struct` to a function copies it, and returning one copies it back

# 2. Member functions
* A function can be defined inside the struct, and is then called as `p.length()`

```
struct vec {
  double x;
  double y;
  double length() {
    return sqrt(x * x + y * y);
  }
};
```

* Inside a member function the members are written without a dot; they refer to the object the function was called on
* (A `class` is the same thing with everything private by default. This course uses `struct`)

# 3. An array inside a struct
* A `struct` may contain an array, and this is how most of the data structures in this course are built

```
struct point_set {
  long n;            // how many points are in use
  double x[8];       // room for 8
  double y[8];
};
```

* The array is _part of_ the struct, not a reference to one somewhere else. `sizeof(point_set)` includes all 16 `double`s, and the whole thing is one contiguous block of memory
* This has a consequence worth knowing: copying the struct copies the arrays with it, element by element. A bare array cannot be assigned at all (`double a[8], b[8]; a = b;` is an error), but wrapped in a struct it can
* The price is that the size must be fixed when the type is written. Later notebooks show the two ways out: making it a compile-time constant with `-D`, and making it a template parameter


In [ ]:
%%writefile vec.cc
#include <stdio.h>
#include <math.h>

struct vec {
  double x;
  double y;
  double length() {
    return sqrt(x * x + y * y);
  }
};

/* move v by (dx, dy) and return the result */
vec shift(vec v, double dx, double dy) {
  v.x += dx;
  v.y += dy;
  return v;
}

struct point_set {
  long n;
  double x[8];
  double y[8];
};

int main() {
  vec p = { 3.0, 4.0 };
  printf("p = (%f, %f), length = %f\n", p.x, p.y, p.length());

  vec q = shift(p, 1.0, 1.0);
  printf("q = (%f, %f)\n", q.x, q.y);
  printf("p = (%f, %f)\n", p.x, p.y);

  point_set s;
  s.n = 3;
  for (long i = 0; i < s.n; i++) {
    s.x[i] = i;
    s.y[i] = 10 * i;
  }
  point_set t = s;          /* copies n and both arrays */
  t.x[0] = 999;
  printf("sizeof(point_set) = %ld\n", sizeof(point_set));
  printf("s.x[0] = %f, t.x[0] = %f\n", s.x[0], t.x[0]);
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -Wall vec.cc -o vec

In [ ]:
%%bash
./vec

* Notice that `shift` modified its parameter `v` and `p` in `main` was not affected, and that `t.x[0] = 999` did not disturb `s`

* <font color="green">Exercise: (a) explain why `p` is unchanged after `shift`, and why `s.x[0]` is unchanged after `t.x[0] = 999`. (b) add a member function `double dot(vec b)` to `vec`, returning `x * b.x + y * b.y`, and call it as `p.dot(q)`. (c) add a member function `double sum_x()` to `point_set` returning the sum of the first `n` elements of `x`. (d) check `sizeof(point_set)` against what you would predict from its members.</font>

* Start from this copy, so the program above stays around to compare with


In [ ]:
%%writefile vec_ex.cc
#include <stdio.h>
#include <math.h>

struct vec {
  double x;
  double y;
  double length() {
    return sqrt(x * x + y * y);
  }
  /* (b) double dot(vec b) { ... } */
};

struct point_set {
  long n;
  double x[8];
  double y[8];
  /* (c) double sum_x() { ... } */
};

int main() {
  vec p = { 3.0, 4.0 };
  vec q = { 1.0, 2.0 };
  printf("p = (%f, %f), q = (%f, %f)\n", p.x, p.y, q.x, q.y);
  /* (b) printf("p.dot(q) = %f\n", p.dot(q)); */

  point_set s;
  s.n = 3;
  for (long i = 0; i < s.n; i++) {
    s.x[i] = i;
    s.y[i] = 10 * i;
  }
  /* (c) printf("s.sum_x() = %f\n", s.sum_x()); */

  /* (d) */
  printf("sizeof(long) = %ld, sizeof(double) = %ld, sizeof(point_set) = %ld\n",
         sizeof(long), sizeof(double), sizeof(point_set));
  return 0;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang++ -Wall vec_ex.cc -o vec_ex

In [ ]:
%%bash
./vec_ex